[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C50_HuggingFace_Ecosystem_Course/01_transformers_core/01_transformers_core.ipynb)

# 01 · transformers 核心抽象（迷你复刻 Auto* / from_pretrained / generate）

目标：把 **config-model-tokenizer 三件套 → from_pretrained 的六步 → 权重键名匹配 → dtype/显存账 → generate 的采样逻辑** 从零写一遍，
并用 assert 钉死那些**静默失败**的坑。

路线：三件套与同源检查 → 六步加载与 loading_info → 键名匹配的三类差异 → 显存账 →
labels 右移的不对称 → generate 的解码策略 → 左 padding → ✏️ 练习 → 📖 答案 → 🧪 上下文预算胶囊。

> 心智模型：**这一层的坑几乎全是「静默失败」**——不报错、能跑、结果错。
> 所以要带走的不是参数表，而是一套**显式检查的习惯**。

## 1 · 三件套与「同源检查」

`config` 是纯数据、`model` 由 config 完全确定、`tokenizer` 与 model **强绑定**。
先把三件套写出来，再实现一个能挡住「tokenizer/model 不匹配」的检查。

In [ ]:
import numpy as np, math, json, hashlib, random
rng = np.random.default_rng(0)

class Config:
    '''纯数据。model_type 是 Auto* 分发的依据。'''
    def __init__(self, model_type, vocab_size, hidden_size, num_layers, num_labels=2, **kw):
        self.model_type, self.vocab_size = model_type, vocab_size
        self.hidden_size, self.num_layers, self.num_labels = hidden_size, num_layers, num_labels
        for k, v in kw.items(): setattr(self, k, v)
    def to_dict(self): return dict(self.__dict__)
    @classmethod
    def from_dict(cls, d):
        d = dict(d); return cls(d.pop('model_type'), d.pop('vocab_size'),
                                d.pop('hidden_size'), d.pop('num_layers'), **d)

class Tokenizer:
    def __init__(self, name, vocab):
        self.name, self.vocab = name, vocab
        self.itos = {i: s for s, i in vocab.items()}
    @property
    def vocab_size(self): return len(self.vocab)
    def __call__(self, text):
        return [self.vocab.get(w, self.vocab['[UNK]']) for w in text.lower().split()]
    def decode(self, ids): return ' '.join(self.itos.get(i, '[UNK]') for i in ids)

class Model:
    def __init__(self, config, name='<random>'):
        self.config, self.name = config, name
        r = np.random.default_rng(abs(hash(name)) % (2**32))
        self.params = {f'layer.{l}.weight': r.normal(size=(config.hidden_size, config.hidden_size)) * 0.02
                       for l in range(config.num_layers)}
        self.params['embeddings.weight'] = r.normal(size=(config.vocab_size, config.hidden_size)) * 0.02
        self.params['classifier.weight'] = r.normal(size=(config.hidden_size, config.num_labels)) * 0.02

VOCAB_A = {w: i for i, w in enumerate(['[UNK]', '[CLS]', '[SEP]', 'good', 'bad', 'movie', 'film'])}
VOCAB_B = {w: i for i, w in enumerate(['[UNK]', '[CLS]', '[SEP]', 'film', 'movie', 'bad', 'good'])}
tok_a, tok_b = Tokenizer('model-a', VOCAB_A), Tokenizer('model-b', VOCAB_B)
cfg = Config('bert', vocab_size=len(VOCAB_A), hidden_size=8, num_layers=2)

text = 'good movie'
print(f'tok_a("{text}") = {tok_a(text)}  -> 解回: {tok_a.decode(tok_a(text))}')
print(f'tok_b("{text}") = {tok_b(text)}  -> 解回: {tok_b.decode(tok_b(text))}')
# 两个词表**大小相同、id 都合法**，但含义完全不同
ids_a = tok_a(text)
print(f'\n⚠️  把 tok_a 的 id {ids_a} 交给 model-b 的词表解读 -> "{tok_b.decode(ids_a)}"')
assert tok_a.vocab_size == tok_b.vocab_size, '词表大小相同 -> 不会有任何越界报错'
assert tok_a(text) != tok_b(text), '但同一句话的 id 完全不同'
assert tok_b.decode(tok_a(text)) != text, '交叉使用会得到彻底的乱码'
print('   代码完全不报错、前向能跑、loss 会降 —— 但模型看到的是乱码。')

In [ ]:
def load_pair(checkpoint, registry):
    '''从**同一个变量**取名字加载 tokenizer 与 model —— 这就是防错的全部秘诀。'''
    tok, cfg_ = registry[checkpoint]
    model = Model(cfg_, name=checkpoint)
    # 显式同源检查
    assert tok.name == checkpoint, f'tokenizer 来源 {tok.name} != {checkpoint}'
    assert model.config.vocab_size == tok.vocab_size, \
        f'词表大小不一致: model {model.config.vocab_size} vs tokenizer {tok.vocab_size}'
    return tok, model

REGISTRY = {
    'model-a': (tok_a, Config('bert', len(VOCAB_A), 8, 2)),
    'model-b': (tok_b, Config('bert', len(VOCAB_B), 8, 2)),
}
CKPT = 'model-a'                      # ← 只写一次，绝不手写两遍
t, m = load_pair(CKPT, REGISTRY)
print(f'加载成功: tokenizer={t.name}, model.vocab_size={m.config.vocab_size}')

# 反例：词表大小对不上时必须报错
bad_reg = {'x': (tok_a, Config('bert', vocab_size=999, hidden_size=8, num_layers=2))}
try:
    load_pair('x', bad_reg); raise RuntimeError('不该到这')
except AssertionError as e:
    print(f'✅ 同源检查拦住了: {e}')
print('\n✅ 铁律：**把 checkpoint 名字写成一个变量**，tokenizer 与 model 都从它加载。')

## 2 · from_pretrained 的六步与键名匹配

第 ⑤ 步（权重键名匹配）是**唯一会静默出问题**的地方。它产生三类差异：
`missing_keys`（模型要但文件没有 → **随机初始化**）、`unexpected_keys`（文件有但模型不要 → 丢弃）、
`mismatched_keys`（名字对但形状不对 → 报错，除非显式忽略）。

In [ ]:
# 模拟 Hub 上的一个「预训练 checkpoint」：有 MLM 头，没有分类头
PRETRAINED = {
    'embeddings.weight':  np.full((7, 8), 1.0),
    'layer.0.weight':     np.full((8, 8), 2.0),
    'layer.1.weight':     np.full((8, 8), 3.0),
    'mlm_head.weight':    np.full((8, 7), 4.0),      # 下游分类任务不需要它
}

def from_pretrained(config, state_dict, ignore_mismatched_sizes=False, output_loading_info=False):
    '''复刻第 ④~⑤ 步：先按 config 建结构（随机），再用 state_dict 覆盖能对上的键。'''
    model = Model(config, name='<random>')
    model_keys, file_keys = set(model.params), set(state_dict)
    missing = sorted(model_keys - file_keys)
    unexpected = sorted(file_keys - model_keys)
    mismatched, loaded = [], []
    for k in sorted(model_keys & file_keys):
        if model.params[k].shape != state_dict[k].shape:
            mismatched.append((k, tuple(state_dict[k].shape), tuple(model.params[k].shape)))
            if not ignore_mismatched_sizes:
                raise RuntimeError(
                    f'size mismatch for {k}: checkpoint {state_dict[k].shape} '
                    f'vs model {model.params[k].shape}. 传 ignore_mismatched_sizes=True 可跳过')
        else:
            model.params[k] = state_dict[k].copy(); loaded.append(k)
    info = {'missing_keys': missing, 'unexpected_keys': unexpected,
            'mismatched_keys': mismatched, 'loaded_keys': loaded}
    return (model, info) if output_loading_info else model

cfg3 = Config('bert', vocab_size=7, hidden_size=8, num_layers=2, num_labels=3)
model, info = from_pretrained(cfg3, PRETRAINED, output_loading_info=True)
for k, v in info.items():
    print(f'{k:<18s}: {v}')

assert info['missing_keys'] == ['classifier.weight'], '分类头是新的 -> 随机初始化（**这是期望行为**）'
assert info['unexpected_keys'] == ['mlm_head.weight'], '预训练的 MLM 头被丢弃'
assert 'layer.0.weight' in info['loaded_keys']
assert model.params['layer.0.weight'][0, 0] == 2.0, '主干权重必须真的被加载了'
assert abs(model.params['classifier.weight'][0, 0]) < 1.0, '分类头是小随机数'
print('\n✅ 微调时看到「分类头随机初始化」是正常的；')
print('   但若 missing_keys 里出现大量 layer.* ，说明键名对不上 —— 模型基本等于随机初始化。')

In [ ]:
def assert_loaded_properly(info, allowed_missing_prefixes=('classifier',)):
    '''把「读日志」变成「断言」——这一个函数能挡住一整类难查的 bug。'''
    bad = [k for k in info['missing_keys']
           if not any(k.startswith(p) for p in allowed_missing_prefixes)]
    assert not bad, f'意外的 missing_keys（主干权重没加载上！）: {bad}'
    assert not info['mismatched_keys'], f'形状不匹配: {info["mismatched_keys"]}'
    return True

assert assert_loaded_properly(info)
print('✅ 正常加载通过检查')

# 反例：模型类选错，键名前缀完全不同
WRONG_PREFIX = {k.replace('layer.', 'encoder.layer.'): v for k, v in PRETRAINED.items()}
_, info_bad = from_pretrained(cfg3, WRONG_PREFIX, output_loading_info=True)
print(f'\n键名前缀不匹配时 missing_keys = {info_bad["missing_keys"]}')
try:
    assert_loaded_properly(info_bad); raise RuntimeError('不该到这')
except AssertionError as e:
    print(f'✅ 断言拦住了: {str(e)[:90]}…')
print('\n真实库里对应: model, info = AutoModel.from_pretrained(..., output_loading_info=True)')

# mismatched：改了 num_labels
cfg5 = Config('bert', vocab_size=7, hidden_size=8, num_layers=2, num_labels=5)
STORE_WITH_HEAD = dict(PRETRAINED); STORE_WITH_HEAD['classifier.weight'] = np.zeros((8, 3))
try:
    from_pretrained(cfg5, STORE_WITH_HEAD); raise RuntimeError('不该到这')
except RuntimeError as e:
    print(f'\n改 num_labels(3->5) 时: {str(e)[:100]}…')
m5, i5 = from_pretrained(cfg5, STORE_WITH_HEAD, ignore_mismatched_sizes=True, output_loading_info=True)
assert i5['mismatched_keys'] and m5.params['classifier.weight'].shape == (8, 5)
print('✅ ignore_mismatched_sizes=True 允许「主干加载 + 头重初始化」——微调的标准情形')

## 3 · 显存账：为什么全量微调 7B 要 84GB

$$\text{推理} \approx N \cdot b_{dtype} \cdot 1.2 \qquad
\text{训练} \approx N\cdot 2 + N\cdot 2 + N_{trainable}\cdot 8 + \text{激活}$$

注意训练的第三项用的是 **可训练参数量**——这正是 LoRA 省显存的根本原因（模块 04）。

In [ ]:
BYTES = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}

def inference_gb(n_params, dtype='fp16', overhead=1.2):
    return n_params * BYTES[dtype] * overhead / 1e9

def kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch=1, dtype='fp16'):
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * BYTES[dtype] / 1e9

def training_gb(n_params, n_trainable=None, weight_dtype='fp16', activations_gb=4.0):
    n_trainable = n_params if n_trainable is None else n_trainable
    w = n_params * BYTES[weight_dtype]
    g = n_trainable * BYTES[weight_dtype]
    opt = n_trainable * 8                 # Adam: fp32 的 m 与 v
    return (w + g + opt) / 1e9 + activations_gb

print(f"{'模型':<16s} {'dtype':>6s} {'权重GB':>8s} {'KV(4k)GB':>9s} {'推理合计':>9s}")
for name, n, nl, nkv, hd in [('Llama-3-8B', 8.0e9, 32, 8, 128),
                             ('Llama-3-70B', 70.0e9, 80, 8, 128)]:
    for dt in ['fp16', 'int4']:
        kv = kv_cache_gb(nl, nkv, hd, 4096, 1, 'fp16')
        print(f'{name:<16s} {dt:>6s} {inference_gb(n, dt):>8.1f} {kv:>9.2f} {inference_gb(n,dt)+kv:>9.1f}')

assert inference_gb(8e9, 'fp16') < 21 and inference_gb(8e9, 'int4') < 6
assert inference_gb(70e9, 'int4') < 50, '70B int4 能进单张 80G 卡'
print(f'\n训练 7B（全量）: {training_gb(7e9):.0f} GB   <- 单卡放不下')
print(f'训练 7B（LoRA, 可训练 0.1%）: {training_gb(7e9, n_trainable=7e6):.0f} GB   <- 单卡可以')
assert training_gb(7e9) > 70 and training_gb(7e9, n_trainable=7e6) < 35
print('✅ 优化器状态只按**可训练参数**算 —— 这就是 LoRA 的省显存原理（模块 04 展开）')

## 4 · labels 的右移：两个方向相反的不对称

- `AutoModelForCausalLM`：你传 `labels = input_ids`，**库内部自动右移**。自己先移一遍 = 移了两位。
- `AutoModelForSeq2SeqLM`：`labels` **不需要**你右移（库据它生成 `decoder_input_ids`）；
  但你若手动传了 `decoder_input_ids`，库就**不再**帮你移。

用 assert 把这两个契约钉死。

In [ ]:
def causal_lm_loss(logits, labels):
    '''复刻 AutoModelForCausalLM 的内部右移：logits[:-1] 对 labels[1:]'''
    shift_logits = logits[:-1]
    shift_labels = labels[1:]
    m = shift_labels != -100
    if m.sum() == 0: return 0.0
    lp = shift_logits - shift_logits.max(-1, keepdims=True)
    lp = lp - np.log(np.exp(lp).sum(-1, keepdims=True))
    return float(-lp[np.arange(len(shift_labels))[m], shift_labels[m]].mean())

V, L = 10, 6
rng2 = np.random.default_rng(1)
ids = rng2.integers(0, V, size=L)
logits = rng2.normal(size=(L, V))
# 让 logits 在「预测下一个」的位置上正确
correct = np.full((L, V), -5.0)
for t in range(L - 1):
    correct[t, ids[t + 1]] = 5.0
correct[L - 1, ids[L - 1]] = 5.0

loss_right = causal_lm_loss(correct, ids)                    # ✅ 传原始 ids
manually_shifted = np.concatenate([ids[1:], [-100]])         # ❌ 自己先移了一遍
loss_double = causal_lm_loss(correct, manually_shifted)
print(f'labels=input_ids（正确）      : loss = {loss_right:.4f}')
print(f'labels=手动右移后（错误）      : loss = {loss_double:.4f}')
assert loss_right < 0.05, '正确用法下 loss 应接近 0'
assert loss_double > loss_right * 10, '自己再移一遍 -> 学的是「预测下下个 token」'
print('\n⚠️  症状：loss 能降、不报错，但生成完全不通顺。')
print('✅ CausalLM: `labels = input_ids`，**不要自己移**。')

def seq2seq_prepare(labels, bos=0):
    '''复刻 Seq2SeqLM：据 labels 生成 decoder_input_ids（右移并前置 bos）。'''
    return np.concatenate([[bos], labels[:-1]])

tgt = np.array([3, 4, 5, 2])
dec_in = seq2seq_prepare(tgt)
print(f'\nSeq2Seq: labels={tgt.tolist()} -> 库自动生成 decoder_input_ids={dec_in.tolist()}')
assert dec_in[0] == 0 and dec_in[1:].tolist() == tgt[:-1].tolist()
print('✅ Seq2SeqLM: labels **不用**你移；但你若手动传 decoder_input_ids，库就不帮你移了。')

## 5 · generate()：解码策略与采样参数的生效条件

**头号误用：设了 `temperature` 却没设 `do_sample=True`** —— 参数被完全忽略，你得到的是贪心。
把三种策略都实现一遍，并验证参数的生效条件。

In [ ]:
def softmax(x):
    x = x - x.max(); e = np.exp(x); return e / e.sum()

def apply_warpers(logits, temperature=1.0, top_k=None, top_p=None):
    '''采样前的 logits 处理（真实库里叫 LogitsWarper）。'''
    lg = logits / max(temperature, 1e-8)
    if top_k:
        thresh = np.sort(lg)[-top_k]
        lg = np.where(lg >= thresh, lg, -np.inf)
    if top_p:
        order = np.argsort(-lg)
        p = softmax(lg[order]); cum = np.cumsum(p)
        keep = order[:max(1, int(np.searchsorted(cum, top_p)) + 1)]
        mask = np.full_like(lg, -np.inf); mask[keep] = lg[keep]; lg = mask
    return lg

def generate(logit_fn, max_new_tokens, do_sample=False, num_beams=1,
             temperature=1.0, top_k=None, top_p=None, eos=None, seed=0):
    '''三种策略：贪心 / 采样 / beam。**采样参数只在 do_sample=True 时生效**。'''
    r = np.random.default_rng(seed)
    if num_beams > 1:
        beams = [([], 0.0)]
        for _ in range(max_new_tokens):
            cands = []
            for seq, sc in beams:
                lg = logit_fn(tuple(seq)); lp = np.log(softmax(lg) + 1e-12)
                for t in np.argsort(lp)[-num_beams:]:
                    cands.append((seq + [int(t)], sc + float(lp[t])))
            beams = sorted(cands, key=lambda x: x[1], reverse=True)[:num_beams]
        return beams[0][0]
    seq = []
    for _ in range(max_new_tokens):
        lg = logit_fn(tuple(seq))
        if do_sample:
            lg = apply_warpers(lg, temperature, top_k, top_p)      # ← 只在这里生效
            t = int(r.choice(len(lg), p=softmax(lg)))
        else:
            t = int(np.argmax(lg))                                  # 贪心：忽略一切采样参数
        seq.append(t)
        if eos is not None and t == eos: break
    return seq

VOC = 8
def make_logit_fn(seed=0):
    def fn(prefix):
        r = np.random.default_rng((hash(prefix) ^ seed) % (2**32))
        return r.normal(size=VOC)
    return fn
fn = make_logit_fn(7)

greedy1 = generate(fn, 6, do_sample=False)
greedy2 = generate(fn, 6, do_sample=False, temperature=0.1, top_p=0.5, seed=99)
print(f'贪心                       : {greedy1}')
print(f'贪心 + temperature/top_p   : {greedy2}   ← 完全相同！参数被忽略了')
assert greedy1 == greedy2, '⚠️ do_sample=False 时，temperature/top_k/top_p 全部无效'

s1 = generate(fn, 6, do_sample=True, temperature=1.0, seed=1)
s2 = generate(fn, 6, do_sample=True, temperature=1.0, seed=2)
print(f'\n采样 seed=1                : {s1}')
print(f'采样 seed=2                : {s2}   ← 不同（有随机性）')
assert s1 != s2, '采样应有随机性'

b = generate(fn, 6, num_beams=3)
print(f'beam=3                     : {b}')
print('\n✅ 这就是最高频的误用：**temperature 只在 do_sample=True 时生效**。')
print('   注意与 OpenAI API 的语义差异：那里 temperature=0 被服务端翻译成贪心，')
print('   而 generate() 里 temperature=0 是非法的（除零）—— 要确定性请用 do_sample=False。')

### 温度与 top-p 的效果：用熵量化

In [ ]:
def entropy_after(logits, **kw):
    return float(-(lambda p: (p * np.log(p + 1e-12)).sum())(softmax(apply_warpers(logits, **kw))))

lg = np.random.default_rng(3).normal(size=32) * 2
print(f'{"设置":<30s} {"熵(nat)":>9s} {"有效候选数":>11s}')
for label, kw in [('原始 (T=1)', dict()),
                  ('T=0.5 (更尖)', dict(temperature=0.5)),
                  ('T=2.0 (更平)', dict(temperature=2.0)),
                  ('top_k=5', dict(top_k=5)),
                  ('top_p=0.9', dict(top_p=0.9)),
                  ('T=0.7 + top_p=0.9', dict(temperature=0.7, top_p=0.9))]:
    h = entropy_after(lg, **kw)
    print(f'{label:<30s} {h:>9.3f} {math.exp(h):>11.2f}')

h1, h_low, h_high = entropy_after(lg), entropy_after(lg, temperature=0.5), entropy_after(lg, temperature=2.0)
assert h_low < h1 < h_high, '温度越低分布越尖（熵越小），越高越平'
assert entropy_after(lg, top_k=5) < h1, 'top-k 截断降低熵'
assert entropy_after(lg, top_p=0.9) < h1, 'top-p 截断降低熵'
print('\n✅ exp(熵) 是「有效候选数」的直观度量。T 调分布形状，top-k/top-p 做硬截断。')
print('   实践：先用 top_p=0.9 截尾（去掉长尾垃圾），再用 T 微调多样性。')

## 6 · 批量生成必须左 padding

decoder-only 从序列末尾续写。**右 padding 会把 pad token 夹在 prompt 与生成之间。**

In [ ]:
PAD = -1

def pad_batch(seqs, side='right', pad=PAD):
    L = max(len(s) for s in seqs)
    out, mask = [], []
    for s in seqs:
        n = L - len(s)
        if side == 'right': out.append(list(s) + [pad] * n); mask.append([1]*len(s) + [0]*n)
        else:              out.append([pad] * n + list(s)); mask.append([0]*n + [1]*len(s))
    return np.array(out), np.array(mask)

prompts = [[1, 2, 3], [4, 5], [6]]
for side in ['right', 'left']:
    ids_, mask_ = pad_batch(prompts, side)
    last_real = [int(np.where(m == 1)[0][-1]) for m in mask_]
    print(f'{side:>5s} padding:')
    for row, lr in zip(ids_, last_real):
        print(f'   {row.tolist()}   最后一个真实 token 在下标 {lr}')

_, mask_left = pad_batch(prompts, 'left')
_, mask_right = pad_batch(prompts, 'right')
left_last = [int(np.where(m == 1)[0][-1]) for m in mask_left]
right_last = [int(np.where(m == 1)[0][-1]) for m in mask_right]
assert len(set(left_last)) == 1, '左 padding: 所有序列的「最后一个真实 token」都在同一列 -> 可以统一续写'
assert len(set(right_last)) > 1, '右 padding: 位置各不相同 -> 续写会从 pad 之后开始'
print(f'\n左 padding 的最后真实位置: {left_last}  ← 全部相同 ✅')
print(f'右 padding 的最后真实位置: {right_last}  ← 各不相同 ❌')
print('\n⚠️  症状：**单条生成正常，批量生成变差** —— 极其常见且容易归因错误。')
print('✅ 批量生成前：tokenizer.padding_side = "left"')
print('   注意：**训练时用右 padding**（labels 对齐更自然），只有生成才要左 padding。')

## ✏️ 练习 1：显存可行性判断

实现 `fits_in_gpu(n_params, gpu_gb, mode, dtype='fp16', n_trainable=None, activations_gb=4.0)`：
`mode ∈ {'inference','train'}`，返回 `(是否放得下, 需要的GB)`。
推理用 `inference_gb`，训练用 `training_gb`。

In [ ]:
def fits_in_gpu(n_params, gpu_gb, mode, dtype='fp16', n_trainable=None, activations_gb=4.0):
    # TODO: 按 mode 选公式；返回 (need <= gpu_gb, need)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ok, need = fits_in_gpu(8e9, 24, 'inference', 'fp16')
assert ok and 18 < need < 22, f'8B fp16 推理约 19GB，得到 {need:.1f}'
ok2, _ = fits_in_gpu(70e9, 80, 'inference', 'fp16')
assert not ok2, '70B fp16 放不进单张 80G'
ok3, _ = fits_in_gpu(70e9, 80, 'inference', 'int4')
assert ok3, '70B int4 可以'
ok4, need4 = fits_in_gpu(7e9, 80, 'train')
assert not ok4, f'7B 全量训练需要 {need4:.0f}GB，单张 80G 放不下'
ok5, need5 = fits_in_gpu(7e9, 80, 'train', n_trainable=7e6)
assert ok5, f'7B LoRA 训练只需 {need5:.0f}GB'
print(f'8B fp16 推理 {need:.1f}GB | 7B 全量训练 {need4:.0f}GB | 7B LoRA 训练 {need5:.0f}GB')
print('✅ 练习 1 通过：这个函数应该成为你每次选型的第一步')

## ✏️ 练习 2：解码配置校验

实现 `validate_generation_config(cfg)`：接收一个 dict，返回**警告列表**（字符串）。要检出：
1. 设了 `temperature`/`top_k`/`top_p` 之一但 `do_sample` 不为 True → `'sampling params ignored'`
2. `num_beams > 1` 且 `do_sample` 为 True → `'beam sample is rarely what you want'`
3. 设了 `length_penalty` 但 `num_beams <= 1` → `'length_penalty ignored'`
4. 用了 `max_length` 而没用 `max_new_tokens` → `'prefer max_new_tokens'`
5. `temperature == 0` → `'temperature=0 is invalid; use do_sample=False'`

In [ ]:
def validate_generation_config(cfg):
    # TODO: 返回警告字符串列表（顺序不限）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w = validate_generation_config({'temperature': 0.7, 'max_new_tokens': 128})
assert 'sampling params ignored' in w, w
w2 = validate_generation_config({'do_sample': True, 'temperature': 0.7, 'max_new_tokens': 128})
assert w2 == [], f'正确配置不应有警告，得到 {w2}'
w3 = validate_generation_config({'do_sample': True, 'num_beams': 4, 'max_new_tokens': 64})
assert 'beam sample is rarely what you want' in w3
w4 = validate_generation_config({'length_penalty': 1.0, 'max_new_tokens': 64})
assert 'length_penalty ignored' in w4
w5 = validate_generation_config({'max_length': 512, 'do_sample': False})
assert 'prefer max_new_tokens' in w5
w6 = validate_generation_config({'do_sample': True, 'temperature': 0.0, 'max_new_tokens': 8})
assert 'temperature=0 is invalid; use do_sample=False' in w6
print('✅ 练习 2 通过：把这个校验函数放进你的推理封装里，能挡住最高频的一类误用')

## ✏️ 练习 3：上下文预算

实现 `context_budget(model_max, n_special, reserve_for_generation, prompt_tokens)`：
返回 `(是否放得下, 可用于 prompt 的最大 token 数, 需要截掉多少)`。
若放得下，截掉量为 0。

In [ ]:
def context_budget(model_max, n_special, reserve_for_generation, prompt_tokens):
    # TODO: usable = model_max - n_special - reserve_for_generation
    #       返回 (prompt_tokens <= usable, usable, max(0, prompt_tokens - usable))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ok, usable, cut = context_budget(512, 3, 128, 300)
assert ok and usable == 381 and cut == 0
ok2, usable2, cut2 = context_budget(512, 3, 128, 500)
assert not ok2 and cut2 == 119, f'应截掉 119，得到 {cut2}'
# 生成预留越多，可用 prompt 越短
_, u_small, _ = context_budget(512, 3, 32, 100)
_, u_large, _ = context_budget(512, 3, 256, 100)
assert u_small > u_large
print(f'model_max=512, special=3: 预留 128 -> prompt 上限 {usable}；预留 256 -> 上限 {u_large}')
print('✅ 练习 3 通过：**长输入被静默截断**是线上最隐蔽的一类 bug —— 显式算预算并报警')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fits_in_gpu(n_params, gpu_gb, mode, dtype='fp16', n_trainable=None, activations_gb=4.0):
    if mode == 'inference':
        need = inference_gb(n_params, dtype)
    elif mode == 'train':
        need = training_gb(n_params, n_trainable, dtype, activations_gb)
    else:
        raise ValueError(mode)
    return need <= gpu_gb, need

In [ ]:
# 练习 2 参考答案
def validate_generation_config(cfg):
    w = []
    sampling = [k for k in ('temperature', 'top_k', 'top_p') if k in cfg]
    if sampling and not cfg.get('do_sample'):
        w.append('sampling params ignored')
    if cfg.get('num_beams', 1) > 1 and cfg.get('do_sample'):
        w.append('beam sample is rarely what you want')
    if 'length_penalty' in cfg and cfg.get('num_beams', 1) <= 1:
        w.append('length_penalty ignored')
    if 'max_length' in cfg and 'max_new_tokens' not in cfg:
        w.append('prefer max_new_tokens')
    if cfg.get('temperature') == 0:
        w.append('temperature=0 is invalid; use do_sample=False')
    return w

In [ ]:
# 练习 3 参考答案
def context_budget(model_max, n_special, reserve_for_generation, prompt_tokens):
    usable = model_max - n_special - reserve_for_generation
    return prompt_tokens <= usable, usable, max(0, prompt_tokens - usable)

---
## 🧪 真实 API 对照胶囊（不在本环境运行，可原样复制）

把本模块的每个检查写成一个**可复用的加载函数**。这段代码就是本模块的交付物。

In [ ]:
RECIPE = r'''
import torch
from transformers import (AutoConfig, AutoTokenizer,
                          AutoModelForSequenceClassification)

CKPT = "microsoft/deberta-v3-base"      # ① 只写一次，两边都从它加载
REVISION = "main"                        # 生产上建议钉 commit sha

def load_classifier(ckpt=CKPT, num_labels=3, revision=REVISION):
    tok = AutoTokenizer.from_pretrained(ckpt, revision=revision, use_fast=True)
    model, info = AutoModelForSequenceClassification.from_pretrained(
        ckpt, revision=revision,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,     # ② 允许「主干加载 + 头重初始化」
        torch_dtype="auto",               # ③ 读 config 里的 dtype，别一律 fp32
        low_cpu_mem_usage=True,           # ④ 边加载边放置
        attn_implementation="sdpa",       # ⑤ 用 PyTorch 原生高效注意力
        output_loading_info=True,         # ⑥ 拿到 loading_info 做断言
    )
    # ⑦ 把「读日志」变成「断言」
    bad = [k for k in info["missing_keys"]
           if not k.startswith(("classifier", "pooler", "score"))]
    assert not bad, f"主干权重未加载: {bad[:5]}"
    assert model.config.vocab_size == len(tok), "tokenizer 与 model 词表不一致"
    return tok, model

# 批量生成（decoder-only）的正确姿势
def batch_generate(tok, model, prompts, max_new_tokens=256):
    tok.padding_side = "left"                       # ⑧ 生成必须左 padding
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token               # ⑨ Llama 系没有 pad token
    enc = tok(prompts, return_tensors="pt", padding=True,
              truncation=True, max_length=1024).to(model.device)
    with torch.inference_mode():
        out = model.generate(**enc,                 # ⑩ 一定要展开传（含 attention_mask）
                             max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]        # ⑪ 只解码新生成的部分
    return tok.batch_decode(gen, skip_special_tokens=True)
'''
print(RECIPE)
checks = ['只写一次', 'ignore_mismatched_sizes', 'output_loading_info',
          'padding_side = "left"', 'pad_token = tok.eos_token', '断言']
for c in checks:
    assert c in RECIPE, c
print(f'✅ 这段配方包含本模块的全部 {len(checks)} 项检查 —— 建议直接放进你的项目工具库')

### 小结
- **三件套**：config 是纯数据、model 由 config 完全确定、tokenizer 与 model **强绑定**。把 checkpoint 名字写成变量，两边都从它加载。
- **`from_pretrained` 六步**里，第 ⑤ 步的键名匹配是唯一会静默出问题的地方。用 `output_loading_info=True` + 断言把它显式化。
- **优先 safetensors**（`.bin` 是 pickle，加载会执行任意代码）；`trust_remote_code` 同理。
- **显存账**：推理 ≈ N×dtype×1.2 + KV；训练 ≈ N×2 + N×2 + **N_trainable**×8 + 激活。最后一项用可训练参数量——这是 LoRA 省显存的根源。
- **labels 右移的两个不对称**：CausalLM 内部帮你移（别自己移）；Seq2SeqLM 不用你移（除非你手传 `decoder_input_ids`）。
- **`temperature` 只在 `do_sample=True` 时生效**；批量生成必须**左 padding**（训练用右 padding）。
- 这一层的坑几乎全是**静默失败**——所以要带走的是**显式检查的习惯**，不是参数表。

下一站：**模块 02 · tokenizers 与 datasets** —— 数据进模型之前的那两层，坑同样多。